# Improved Model 3 — FLAN-T5 team classifier on Colab

Trains `src/improved_model_3` with `--per-team-cap 5000` on a Colab GPU. Checkpoints and logs go to your Google Drive so they survive runtime disconnects.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 free, L4/A100 if you have Pro).
2. On first run, cell 2.5 will prompt you to upload the two parquets from your laptop. They'll be copied to Drive and reused thereafter.
3. Make sure the repo is pushed to GitHub on the branch you want to train from.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ClariCase'
DRIVE_SPLITS = f'{DRIVE_ROOT}/splits_v2'
DRIVE_CKPT  = f'{DRIVE_ROOT}/improved_model_3/checkpoints'
DRIVE_LOGS  = f'{DRIVE_ROOT}/improved_model_3/logs'
os.makedirs(DRIVE_SPLITS, exist_ok=True)
os.makedirs(DRIVE_CKPT,  exist_ok=True)
os.makedirs(DRIVE_LOGS,  exist_ok=True)

need = ['train_cap_issue.parquet', 'val.parquet']
missing = [f for f in need if not os.path.exists(f'{DRIVE_SPLITS}/{f}')]
if missing:
    print(f'MISSING from Drive: {missing}')
    print(f'Run the next cell (2.5) to upload them from your laptop.')
else:
    print('drive OK; splits present')

## 2.5. First-run only — upload the parquets from your laptop

Only needed if cell 2 reported missing files. A file-picker will appear; select these two from `F:\NLP_MSML641\data\cache\splits_v2\` on your laptop:

- `train_cap_issue.parquet` (163 MB)
- `val.parquet` (97 MB)

They'll be uploaded into the Colab runtime and copied to Drive, so future runs skip this step. Upload takes several minutes — leave the tab focused.

In [ ]:
import os, shutil
from google.colab import files

need = ['train_cap_issue.parquet', 'val.parquet']
missing = [f for f in need if not os.path.exists(f'{DRIVE_SPLITS}/{f}')]
if not missing:
    print('nothing to upload; both files already in Drive')
else:
    print(f'Upload these files: {missing}')
    uploaded = files.upload()
    for name in uploaded:
        if name in need:
            dst = f'{DRIVE_SPLITS}/{name}'
            shutil.move(name, dst)
            print(f'copied {name} -> {dst}')
        else:
            os.remove(name)
            print(f'ignored unexpected file: {name}')

    still_missing = [f for f in need if not os.path.exists(f'{DRIVE_SPLITS}/{f}')]
    assert not still_missing, f'still missing after upload: {still_missing}'
    print('all files now in Drive')

## 3. Clone the repo

In [ ]:
REPO_URL = 'https://github.com/farcypeer46/ClariCase.git'
BRANCH   = 'improved_model_3'
REPO_DIR = '/content/ClariCase'

import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH,
                    REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard',
                    f'origin/{BRANCH}'], check=True)

os.chdir(REPO_DIR)
print('repo at', REPO_DIR, 'on branch', BRANCH)
!git log -1 --oneline

## 4. Install dependencies

In [ ]:
!pip uninstall -y -q torchao
!pip install -q transformers peft accelerate datasets sentencepiece \
    scikit-learn tqdm pandas pyarrow bitsandbytes python-dotenv

## 5. Wire Drive into the repo's cache layout

In [ ]:
import os
os.makedirs('data/cache', exist_ok=True)
os.makedirs('data/cache/improved_model_3', exist_ok=True)
os.makedirs('logs', exist_ok=True)

def link(src, dst):
    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.islink(dst): os.unlink(dst)
        else:
            import shutil; shutil.rmtree(dst) if os.path.isdir(dst) else os.remove(dst)
    os.symlink(src, dst)
    print('ln', dst, '->', src)

link(DRIVE_SPLITS, 'data/cache/splits_v2')
link(DRIVE_CKPT,   'data/cache/improved_model_3/checkpoints')
link(DRIVE_LOGS,   'logs/improved_model_3')

!ls -la data/cache/splits_v2
!ls -la data/cache/improved_model_3

## 6. Tune config for Colab's bigger GPU

In [ ]:
import re, pathlib, torch

total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU VRAM: {total_vram:.1f} GB')

if total_vram >= 35:   # A100 40GB
    PDB, GAS = 16, 1
elif total_vram >= 20: # L4 24GB, V100 32GB
    PDB, GAS = 8, 2
elif total_vram >= 14: # T4 16GB
    PDB, GAS = 8, 2
else:
    PDB, GAS = 4, 4
print(f'using PER_DEVICE_BATCH_SIZE={PDB}, GRAD_ACCUM_STEPS={GAS}')

cfg = pathlib.Path('src/improved_model_3/config.py')
text = cfg.read_text()
text = re.sub(r'PER_DEVICE_BATCH_SIZE\s*=\s*\d+',
              f'PER_DEVICE_BATCH_SIZE = {PDB}', text)
text = re.sub(r'GRAD_ACCUM_STEPS\s*=\s*\d+',
              f'GRAD_ACCUM_STEPS = {GAS}', text)
cfg.write_text(text)
!grep -E '^(PER_DEVICE_BATCH_SIZE|GRAD_ACCUM_STEPS|EVAL_STEPS|SAVE_STEPS|SAVE_TOTAL_LIMIT|LOGGING_STEPS)\s*=' src/improved_model_3/config.py

## 7. Launch training (5 k per team)

Add `--resume` to pick up the latest checkpoint in Drive. Omit it for a fresh run.

Expected wall-clock (5 k per team, ~50 k rows, 2 epochs):
- **T4 (free)**: ~2–3 h
- **L4**: ~1–1.5 h
- **A100**: ~30–45 min

In [ ]:
import os
RESUME = os.path.isdir('data/cache/improved_model_3/checkpoints') and \
         any(d.startswith('checkpoint-')
             for d in os.listdir('data/cache/improved_model_3/checkpoints'))
resume_flag = '--resume' if RESUME else ''
print('resume?', RESUME)

In [ ]:
!python -u -m src.improved_model_3.train --per-team-cap 5000 $resume_flag \
    2>&1 | tee -a logs/improved_model_3/train_5k_per_team.log

## 8. (Optional) Quick status cells you can run in another kernel instance

In [ ]:
import json, glob
ckpts = sorted(glob.glob('data/cache/improved_model_3/checkpoints/checkpoint-*'),
               key=lambda p: int(p.split('-')[-1]))
print('checkpoints:', [p.split('/')[-1] for p in ckpts])

rows = []
for p in ckpts:
    try:
        ts = json.load(open(f'{p}/trainer_state.json'))
        for x in ts['log_history']:
            if 'eval_team_macro_f1' in x:
                rows.append((p.split('/')[-1], x['step'],
                             round(x['eval_team_macro_f1'], 4),
                             round(x['eval_team_accuracy'], 4)))
    except FileNotFoundError:
        pass
rows.sort(key=lambda r: r[2], reverse=True)
print('top eval metrics (by macro_f1):')
for r in rows[:10]:
    print(' ', r)

## 9. (Optional) Pull a trained checkpoint back to local

After training, download the winning checkpoint folder from `MyDrive/ClariCase/improved_model_3/checkpoints/checkpoint-XXXX` via the Drive web UI, then drop it into your local `data/cache/improved_model_3/checkpoints/` to run `predict.py` on your machine.